In [ ]:
# =============================================================================
# DKTC 위협 대화 분류 — 규칙 기반 후처리(Logit Adjustment) 분석
#
# 근거: dktc_model_improvement_runbook_2026-03-17.md §8.2
#
# 목적:
#   학습된 모델의 추론 시 softmax 확률 기반으로
#   top1-top2 margin이 작은 경계 샘플에만 규칙 점수를 부여하여
#   괴롭힘 내부 클래스 혼동을 완화한다.
#
# 타겟 혼동축 (런북 §4.4, §5.4 기준):
#   - 협박 ↔ 갈취
#   - 기타괴롭힘 ↔ 갈취
#   - 직장내괴롭힘 ↔ 기타괴롭힘
#
# 후처리 원칙:
#   1. 모델이 확신하는 예측(margin > threshold)은 건드리지 않는다.
#   2. 경계 샘플에서만 키워드 기반 보정을 적용한다.
#   3. 보정 강도는 보수적으로 시작한다.
#
# 전제 조건:
#   - focal_epoch_5 (또는 다른 best 모델) 체크포인트가 저장되어 있어야 함
#   - data/train_processed_260317_n_1000.csv, val_processed_260317_n_1000.csv 존재
#
# 환경: Docker (py-gpu-env), RTX 5060 Ti 16GB, Python 3.10
# =============================================================================
 

In [ ]:
# ============================================================================
# 00. 환경 설정 및 최적화
# ============================================================================
import sys, site, os
print("sys.executable =", sys.executable)
print("sys.prefix     =", sys.prefix)
print("site-packages  =", site.getsitepackages())
 
import transformers, datasets, evaluate, huggingface_hub, accelerate
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("accelerate:", accelerate.__version__)
 
 
# %%
import os, time, gc, warnings, json, re
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
 
# 재현성
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
from datasets import Dataset, DatasetDict
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from collections import Counter
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
 
# --- CPU / GPU 최적화 ---
CPU_COUNT = os.cpu_count() or 4
CPU_THREADS = max(1, CPU_COUNT // 2)
 
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
 
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(max(1, min(4, CPU_THREADS // 2)))
except RuntimeError:
    pass
 
USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"
 
if USE_CUDA:
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
 
print("=" * 70)
print(f"PyTorch              : {torch.__version__}")
print(f"CUDA available       : {USE_CUDA}")
print(f"Device               : {DEVICE}")
print(f"CPU count            : {CPU_COUNT}")
print(f"CPU threads          : {CPU_THREADS}")
if USE_CUDA:
    props = torch.cuda.get_device_properties(0)
    free_mem, total_mem = torch.cuda.memory.mem_get_info(0)

    print(f"GPU name             : {props.name}")
    print(f"GPU count            : {torch.cuda.device_count()}")
    print(f"GPU memory(total)    : {props.total_memory / (1024**3):.2f} GB")
    print(f"GPU memory(free)     : {free_mem / (1024**3):.2f} GB")
print("=" * 70)

In [ ]:
# ============================================================================
# 01. 데이터 로드 + 토크나이징
# ============================================================================
MODEL_NAME = "klue/roberta-base"     # 한국어 RoBERTa (111M params)
MAX_LENGTH = 512                     # 97% 커버리지 (전처리 품질 리포트 기준)
NUM_LABELS = 5                       # 협박(0), 갈취(1), 직장내괴롭힘(2), 기타괴롭힘(3), 일반대화(4)
 
LABEL_NAMES = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]
LABEL2ID = {name: idx for idx, name in enumerate(LABEL_NAMES)}
 
# --- 전처리된 CSV 로드 ---
train_df = pd.read_csv("data/train_processed_260317_n_1000.csv")
val_df = pd.read_csv("data/val_processed_260317_n_1000.csv")
 
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Classes: {NUM_LABELS}")
print(f"\n검증 클래스 분포:")
print(val_df["label_name"].value_counts().to_string())
 
# --- 토크나이저 ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
 
# --- Hugging Face Dataset 변환 ---
raw_datasets = DatasetDict({
    "train":      Dataset.from_pandas(train_df[["text", "label"]].rename(columns={"label": "labels"}).reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df[["text", "label"]].rename(columns={"label": "labels"}).reset_index(drop=True)),
})
 
def preprocess_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)
 
tokenized_datasets = raw_datasets.map(
    preprocess_function, batched=True, batch_size=1000,
    num_proc=max(1, min(8, CPU_THREADS)),
    remove_columns=["text"], desc="Tokenizing"
)
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
 
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if USE_CUDA else None,
    return_tensors="pt",
)
 
# --- 원본 텍스트 배열 (규칙 적용에 필요) ---
val_texts = val_df["text"].values
val_true_labels = val_df["label"].values
 
print(f"\n토크나이징 완료 — 원본 텍스트 {len(val_texts)}개 보존")

In [ ]:
# ============================================================================
# 02. 모델 로드 + Raw Logit 추출
# ============================================================================
# 사용할 체크포인트 경로 설정
# 실제 환경에 맞게 아래 경로를 수정할 것
#   예: "./results_dktc_focal_epoch_5/checkpoint-XXXX"
#       또는 저장된 best 모델 디렉토리
 
CHECKPOINT_PATH = "./results_dktc_focal_epoch_5"   # ← 실제 경로로 수정
 
# --- 체크포인트에서 모델 로드 ---
# 참고: from_pretrained는 디렉토리 안의 config.json + model.safetensors를 찾음
#       checkpoint-XXXX 하위 디렉토리일 수도 있으므로 확인 필요
import glob
ckpt_candidates = sorted(glob.glob(os.path.join(CHECKPOINT_PATH, "checkpoint-*")))
if ckpt_candidates:
    model_path = ckpt_candidates[-1]   # 가장 마지막 체크포인트
    print(f"체크포인트 발견: {model_path}")
else:
    model_path = CHECKPOINT_PATH
    print(f"직접 경로 사용: {model_path}")
 
model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=NUM_LABELS
)
model.eval()
if USE_CUDA:
    model = model.to(DEVICE)
 
# --- Trainer로 raw logit 추출 ---
# trainer.predict()의 predictions는 raw logits (softmax 전)
# 검증: HF Trainer 소스 — model(**inputs).logits를 그대로 반환
import evaluate as ev
acc_metric = ev.load("accuracy")
f1_metric  = ev.load("f1")
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro":    f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }
 
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
 
# --- 예측 실행 ---
print("\nValidation 추론 시작...")
val_result = trainer.predict(tokenized_datasets["validation"])
 
# raw_logits: shape (N, 5) — softmax 전 값
raw_logits = val_result.predictions
val_labels = val_result.label_ids
 
# --- sanity check: argmax(logits) == argmax(softmax(logits)) ---
# softmax은 단조증가 변환이므로 argmax가 동일해야 함
preds_from_logits = np.argmax(raw_logits, axis=-1)
probs = torch.softmax(torch.tensor(raw_logits, dtype=torch.float32), dim=-1).numpy()
preds_from_probs = np.argmax(probs, axis=-1)
assert np.array_equal(preds_from_logits, preds_from_probs), \
    "FATAL: argmax(logits) != argmax(softmax(logits))"
 
# --- 기본 성능 확인 (후처리 전) ---
baseline_preds = preds_from_logits.copy()
baseline_acc = np.mean(baseline_preds == val_labels)
baseline_f1_macro = f1_score(val_labels, baseline_preds, average="macro")
baseline_f1_weighted = f1_score(val_labels, baseline_preds, average="weighted")
 
print(f"\n후처리 전 성능 (모델 원본):")
print(f"  Accuracy      : {baseline_acc:.4f}")
print(f"  Macro F1      : {baseline_f1_macro:.4f}")
print(f"  Weighted F1   : {baseline_f1_weighted:.4f}")
print(f"  오분류 건수    : {(baseline_preds != val_labels).sum()} / {len(val_labels)}")
print(f"\nLogit shape: {raw_logits.shape}")
print(f"Prob 합계 검증 (첫 5개): {probs[:5].sum(axis=1)}")
 
# --- 메모리 해제 (모델은 더 이상 필요 없음) ---
del model, trainer
gc.collect()
if USE_CUDA:
    torch.cuda.empty_cache()
 
print("\nLogit 추출 완료 — 모델 메모리 해제")

In [ ]:
# ============================================================================
# 03. 경계 샘플 분석 — Margin 분포 확인
# ============================================================================
# margin = top1_prob - top2_prob
# margin이 작을수록 모델이 두 클래스를 헷갈리고 있다는 뜻
 
sorted_probs = np.sort(probs, axis=-1)[:, ::-1]   # 내림차순
margins = sorted_probs[:, 0] - sorted_probs[:, 1]  # top1 - top2
 
# --- 통계 ---
print("Margin 분포 통계:")
print(f"  mean    : {margins.mean():.4f}")
print(f"  median  : {np.median(margins):.4f}")
print(f"  std     : {margins.std():.4f}")
print(f"  min     : {margins.min():.4f}")
print(f"  max     : {margins.max():.4f}")
 
# --- 구간별 샘플 수와 정확도 ---
thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
print(f"\n{'threshold':>10} | {'count':>6} | {'비율':>7} | {'오분류수':>7} | {'구간 오분류율':>12}")
print("-" * 60)
for th in thresholds:
    mask = margins < th
    n = mask.sum()
    if n > 0:
        n_wrong = (baseline_preds[mask] != val_labels[mask]).sum()
        err_rate = n_wrong / n
    else:
        n_wrong = 0
        err_rate = 0.0
    print(f"  < {th:.2f}   | {n:6d} | {n/len(margins)*100:6.1f}% | {n_wrong:7d} | {err_rate*100:10.1f}%")
 
# --- 경계 샘플의 혼동 패턴 분석 ---
# margin < 0.20 인 샘플에서 top1/top2 클래스 쌍 분석
ANALYSIS_THRESHOLD = 0.20
boundary_mask = margins < ANALYSIS_THRESHOLD
print(f"\n분석 대상: margin < {ANALYSIS_THRESHOLD} → {boundary_mask.sum()}개 ({boundary_mask.sum()/len(margins)*100:.1f}%)")
 
if boundary_mask.sum() > 0:
    top2_classes = np.argsort(probs, axis=-1)[:, -2:][:, ::-1]  # (N, 2): [top1, top2]
    boundary_top1 = top2_classes[boundary_mask, 0]
    boundary_top2 = top2_classes[boundary_mask, 1]
    boundary_true = val_labels[boundary_mask]
    boundary_preds = baseline_preds[boundary_mask]
 
    # top1↔top2 혼동 쌍 빈도
    pair_counter = Counter()
    for t1, t2 in zip(boundary_top1, boundary_top2):
        pair = tuple(sorted([t1, t2]))   # 순서 무관하게 집계
        pair_counter[pair] += 1
 
    print(f"\n경계 샘플 top1↔top2 혼동 쌍 (상위 10):")
    for (c1, c2), cnt in pair_counter.most_common(10):
        print(f"  {LABEL_NAMES[c1]:10s} ↔ {LABEL_NAMES[c2]:10s}: {cnt}건")
 
    # 경계 샘플 중 오분류 건
    boundary_wrong = boundary_preds != boundary_true
    print(f"\n경계 샘플 오분류: {boundary_wrong.sum()} / {len(boundary_true)} "
          f"({boundary_wrong.mean()*100:.1f}%)")
 
    if boundary_wrong.sum() > 0:
        print("\n경계 샘플 오분류 패턴 (실제→예측):")
        wrong_pairs = list(zip(boundary_true[boundary_wrong], boundary_preds[boundary_wrong]))
        for (true_cls, pred_cls), cnt in Counter(wrong_pairs).most_common(10):
            print(f"  {LABEL_NAMES[true_cls]:10s} → {LABEL_NAMES[pred_cls]:10s}: {cnt}건")

In [ ]:
# ============================================================================
# 04. 키워드 사전 정의
# ============================================================================
# 원칙: 보수적으로 시작. 정밀도 > 재현율.
# 한 클래스에 과도한 키워드를 넣으면 false positive가 증가하므로
# 가능한 각 클래스에 고유하고 구별력 높은 표현만 사용.
#
# 키워드 매칭 방식: 정규표현식 기반 부분 일치 (형태소 분석 없이)
# 이유: 형태소 분석기 의존성 최소화 + 대회 추론 환경 단순화
#
# 주의사항:
#   - 키워드는 해당 클래스에서 **다른 클래스와 구별되는** 표현이어야 함
#   - 예: "돈"은 갈취에도 나올 수 있지만 일상대화에서도 나올 수 있음
#         → 갈취 맥락과 결합된 패턴으로 제한
#   - 아래 사전은 1차 버전이며, 검증 후 반복 개선 필요
 
# --- 클래스별 신호 키워드 ---
# 각 항목은 re.search()에서 사용할 정규표현식 패턴
KEYWORD_RULES = {
    # 협박(0): 해악 예고 표현
    # 구별 포인트: 갈취와 달리 '대가/요구' 없이 직접적 해악을 예고
    "협박": [
        r"죽[이여일]|죽을래",          # 죽이다/죽여/죽일/죽을래
        r"가만.*안.*[두둬]|두고.*봐",     # 가만 안 둔다/둬 / 두고 봐라
        r"혼[내나]|본때",              # 혼내주다 / 본때를 보여주다
        r"해코지|보복",                # 해코지 / 보복
        r"때려|패[줘주]|맞을래",        # 물리적 폭력 예고
        r"불태[우워]|부[숴셔]",         # 재물 손괴 위협
        r"찾아간다|찾아갈",            # 직접 찾아가겠다는 위협
        r"신고.*할|경찰.*부[를를]",     # 신고 위협 (협박 맥락)
        r"가족.*건[들드]|부모.*알",     # 가족 대상 위협
    ],
 
    # 갈취(1): 요구/대가/금전/행동 강요 표현
    # 구별 포인트: 협박과 달리 '무엇을 내놓으라/해라'는 요구가 핵심
    "갈취": [
        r"돈.*[내줘놔]|[내줘놔].*돈",    # 돈 관련 요구
        r"만원|천원|백만|입금",          # 금액/입금
        r"[내놔줘].*가져|가져.*와",      # 내놔/가져와
        r"빌려|갚[아어]",               # 금전 대차
        r"대가|보상|배상",              # 대가 요구
        r"사[줘주]|쏴[줘주]|한턱",       # 강제 지출
        r"카드.*[줘놔]|계좌.*[보알]",    # 금융정보 요구
        r"물건.*[내놔줘]",              # 물건 갈취
    ],
 
    # 직장내괴롭힘(2): 회사/상사/업무/평가/보고 문맥
    # 구별 포인트: 기타괴롭힘과 달리 직장/조직 맥락이 명시적
    "직장내괴롭힘": [
        r"회사|직장|사무실",            # 직장 공간
        r"상사|부[장하]|과장|팀장|대[리표]|사[장수원]",  # 직급/직위
        r"업무|보고|회의|프로젝트",      # 업무 맥락
        r"승진|평가|인사|발령",          # 인사 관련
        r"퇴사|사직|해고|짤[려리]",      # 고용 위협
        r"야근|출근|퇴근|근무",          # 근로 맥락
        r"직[원급속]|동[료기]|선[배임]|후[배임]",  # 직장 관계
    ],
 
    # 기타괴롭힘(3): 일반적 괴롭힘 (직장 외)
    # 주의: 이 클래스는 '나머지 괴롭힘' 성격이라 키워드 정의가 가장 어려움
    # 전략: 직접 키워드보다, 다른 클래스 키워드가 '없는' 괴롭힘으로 판별
    # 여기서는 기타괴롭힘에 고유한 패턴만 제한적으로 정의
    "기타괴롭힘": [
        r"따돌[려리림]",                # 따돌림/왕따
        r"왕따|은따|전따",              # 따돌림 유형
        r"무시.*[해하]|비[웃꼬][어으]",  # 무시/비웃음
        r"소문.*[퍼내]|험담",            # 소문/험담
        r"창피|망신|수치",              # 수치심 유발
        r"놀[려리림]|조[롱롱]",          # 놀림/조롱
    ],
 
    # 일반대화(4): 별도 규칙 불필요
    # 이유: F1 0.9899로 이미 매우 강함 (런북 §4.3, §5.4)
    # 오히려 일반대화에 규칙을 추가하면 다른 클래스를 잘못 일반대화로 바꿀 위험
}
 
# --- 키워드 패턴 사전 컴파일 ---
COMPILED_RULES = {}
for class_name, patterns in KEYWORD_RULES.items():
    class_id = LABEL2ID[class_name]
    COMPILED_RULES[class_id] = [re.compile(p) for p in patterns]
 
print("키워드 규칙 정의 완료")
for class_name, patterns in KEYWORD_RULES.items():
    print(f"  {class_name:10s}: {len(patterns)}개 패턴")

In [ ]:
# ============================================================================
# 05. 키워드 매칭 함수 + 규칙 기반 후처리 함수
# ============================================================================
 
def count_keyword_hits(text, class_id):
    """
    주어진 텍스트에서 class_id에 해당하는 키워드 패턴 매칭 횟수를 반환.
    매칭 횟수가 아닌 '매칭된 패턴 수'를 반환함 (동일 패턴 중복 카운트 방지).
    """
    if class_id not in COMPILED_RULES:
        return 0
    hits = sum(1 for p in COMPILED_RULES[class_id] if p.search(text))
    return hits
 
 
def apply_rule_postprocessing(
    logits,                  # (N, 5) raw logits
    texts,                   # (N,) 원본 텍스트
    margin_threshold=0.15,   # 이 값 미만인 경우에만 규칙 적용
    boost_per_hit=0.5,       # 키워드 1개 매칭당 logit 보정 값
    max_boost=1.5,           # 최대 보정 한도 (과보정 방지)
):
    """
    규칙 기반 logit 후처리.
 
    동작 원리:
      1. softmax(logits)에서 top1-top2 margin 계산
      2. margin >= threshold → 원본 예측 유지 (변경 없음)
      3. margin < threshold → 텍스트에서 클래스별 키워드 매칭
      4. 매칭된 키워드가 있는 클래스에 logit 보정값 추가
      5. 보정된 logits에서 argmax로 최종 예측
 
    보정은 raw logit 공간에서 수행함.
    이유: softmax 후 확률을 직접 수정하면 확률 합 != 1 문제가 생김.
          logit 공간에서 보정 후 다시 argmax를 취하면 일관성 유지.
 
    Returns:
        adjusted_preds: (N,) 후처리된 예측 레이블
        adjustment_log: dict — 후처리 통계
    """
    N = len(logits)
    adjusted_logits = logits.copy()  # 원본 보존
 
    # --- softmax 확률 + margin 계산 ---
    probs_t = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    sorted_p = np.sort(probs_t, axis=-1)[:, ::-1]
    margins_arr = sorted_p[:, 0] - sorted_p[:, 1]
 
    # --- 통계 수집용 ---
    n_candidates = 0          # 후처리 대상 (margin < threshold)
    n_adjusted = 0            # 실제 보정이 발생한 샘플
    n_changed = 0             # 예측이 실제로 변경된 샘플
    change_log = []           # (idx, before, after, margin, hits) 기록
 
    original_preds = np.argmax(logits, axis=-1)
 
    for i in range(N):
        # margin이 충분하면 건너뜀
        if margins_arr[i] >= margin_threshold:
            continue
 
        n_candidates += 1
        text = texts[i]
 
        # 각 클래스별 키워드 매칭
        hit_counts = {}
        for class_id in COMPILED_RULES:
            hits = count_keyword_hits(text, class_id)
            if hits > 0:
                hit_counts[class_id] = hits
 
        if not hit_counts:
            continue  # 키워드 매칭 없으면 보정 안 함
 
        n_adjusted += 1
 
        # logit 보정 적용
        for class_id, hits in hit_counts.items():
            boost = min(hits * boost_per_hit, max_boost)
            adjusted_logits[i, class_id] += boost
 
        # 예측 변경 확인
        new_pred = np.argmax(adjusted_logits[i])
        old_pred = original_preds[i]
        if new_pred != old_pred:
            n_changed += 1
            change_log.append({
                "idx": i,
                "before": int(old_pred),
                "after": int(new_pred),
                "true_label": int(val_true_labels[i]) if i < len(val_true_labels) else -1,
                "margin": float(margins_arr[i]),
                "hits": hit_counts,
            })
 
    adjusted_preds = np.argmax(adjusted_logits, axis=-1)
 
    adjustment_log = {
        "margin_threshold": margin_threshold,
        "boost_per_hit": boost_per_hit,
        "max_boost": max_boost,
        "total_samples": N,
        "n_candidates": n_candidates,
        "n_adjusted": n_adjusted,
        "n_changed": n_changed,
        "change_log": change_log,
    }
 
    return adjusted_preds, adjustment_log
 
 
print("후처리 함수 정의 완료")

In [ ]:
# ============================================================================
# 06. 후처리 적용 + 성능 비교
# ============================================================================
# 하이퍼파라미터: 보수적으로 시작
# margin_threshold: 0.15 (경계 샘플의 약 10~20% 정도를 타겟)
# boost_per_hit: 0.5 (logit scale에서 작은 보정)
# max_boost: 1.5 (최대 3개 키워드 매칭까지만 반영)
 
adjusted_preds, adj_log = apply_rule_postprocessing(
    logits=raw_logits,
    texts=val_texts,
    margin_threshold=0.15,
    boost_per_hit=0.5,
    max_boost=1.5,
)
 
# --- 후처리 통계 ---
print("후처리 통계:")
print(f"  전체 샘플       : {adj_log['total_samples']}")
print(f"  후처리 대상     : {adj_log['n_candidates']} (margin < {adj_log['margin_threshold']})")
print(f"  키워드 매칭 발생 : {adj_log['n_adjusted']}")
print(f"  예측 변경 발생   : {adj_log['n_changed']}")
 
# --- 성능 비교 ---
adjusted_acc = np.mean(adjusted_preds == val_labels)
adjusted_f1_macro = f1_score(val_labels, adjusted_preds, average="macro")
adjusted_f1_weighted = f1_score(val_labels, adjusted_preds, average="weighted")
 
print(f"\n{'지표':>15} | {'후처리 전':>10} | {'후처리 후':>10} | {'변화':>10}")
print("-" * 55)
print(f"{'Accuracy':>15} | {baseline_acc:>10.4f} | {adjusted_acc:>10.4f} | {adjusted_acc - baseline_acc:>+10.4f}")
print(f"{'Macro F1':>15} | {baseline_f1_macro:>10.4f} | {adjusted_f1_macro:>10.4f} | {adjusted_f1_macro - baseline_f1_macro:>+10.4f}")
print(f"{'Weighted F1':>15} | {baseline_f1_weighted:>10.4f} | {adjusted_f1_weighted:>10.4f} | {adjusted_f1_weighted - baseline_f1_weighted:>+10.4f}")
print(f"{'오분류 건수':>15} | {(baseline_preds != val_labels).sum():>10d} | {(adjusted_preds != val_labels).sum():>10d} | {(adjusted_preds != val_labels).sum() - (baseline_preds != val_labels).sum():>+10d}")
 
# --- 변경 내역 상세 ---
if adj_log['change_log']:
    print(f"\n예측 변경 상세 ({len(adj_log['change_log'])}건):")
    n_improved = 0
    n_worsened = 0
    n_neutral = 0
 
    for cl in adj_log['change_log']:
        before_name = LABEL_NAMES[cl['before']]
        after_name = LABEL_NAMES[cl['after']]
        true_name = LABEL_NAMES[cl['true_label']] if cl['true_label'] >= 0 else "?"
        was_correct = cl['before'] == cl['true_label']
        now_correct = cl['after'] == cl['true_label']
 
        if not was_correct and now_correct:
            status = "✓ 개선"
            n_improved += 1
        elif was_correct and not now_correct:
            status = "✗ 악화"
            n_worsened += 1
        else:
            status = "— 무관"  # 둘 다 틀리거나, 전/후 모두 맞는 경우
            n_neutral += 1
 
        hit_str = ", ".join(f"{LABEL_NAMES[k]}:{v}" for k, v in cl['hits'].items())
        print(f"  [{cl['idx']:4d}] {before_name:8s} → {after_name:8s} (정답: {true_name:8s}) "
              f"margin={cl['margin']:.3f} hits=[{hit_str}] {status}")
 
    print(f"\n변경 요약: 개선 {n_improved} / 악화 {n_worsened} / 무관 {n_neutral}")
    print(f"순효과: {n_improved - n_worsened:+d}건")

In [ ]:
# ============================================================================
# 07. 클래스별 상세 비교 (classification_report)
# ============================================================================
print("후처리 전:")
print("=" * 70)
print(classification_report(
    val_labels, baseline_preds,
    target_names=LABEL_NAMES, digits=4
))
 
print("\n후처리 후:")
print("=" * 70)
print(classification_report(
    val_labels, adjusted_preds,
    target_names=LABEL_NAMES, digits=4
))
 
# --- 클래스별 F1 변화 ---
from sklearn.metrics import f1_score as sk_f1
print("\n클래스별 F1 변화:")
print(f"{'클래스':>10} | {'전':>8} | {'후':>8} | {'변화':>8}")
print("-" * 42)
for i, name in enumerate(LABEL_NAMES):
    mask = val_labels == i
    before_correct = (baseline_preds[mask] == i).sum()
    after_correct = (adjusted_preds[mask] == i).sum()
    # 개별 클래스 F1을 binary로 계산
    f1_before = sk_f1(val_labels == i, baseline_preds == i, zero_division=0)
    f1_after = sk_f1(val_labels == i, adjusted_preds == i, zero_division=0)
    delta = f1_after - f1_before
    marker = "↑" if delta > 0.0001 else ("↓" if delta < -0.0001 else "=")
    print(f"{name:>10} | {f1_before:>8.4f} | {f1_after:>8.4f} | {delta:>+8.4f} {marker}")

In [ ]:
# ============================================================================
# 08. Confusion Matrix 비교 (Before / After)
# ============================================================================
cm_before = confusion_matrix(val_labels, baseline_preds)
cm_after = confusion_matrix(val_labels, adjusted_preds)
cm_diff = cm_after - cm_before
 
print("Confusion Matrix — 후처리 전:")
print(f"{'':>12}", end="")
for name in LABEL_NAMES:
    print(f"{name:>10}", end="")
print()
for i, name in enumerate(LABEL_NAMES):
    print(f"{name:>12}", end="")
    for j in range(NUM_LABELS):
        print(f"{cm_before[i, j]:>10}", end="")
    print()
 
print(f"\nConfusion Matrix — 후처리 후:")
print(f"{'':>12}", end="")
for name in LABEL_NAMES:
    print(f"{name:>10}", end="")
print()
for i, name in enumerate(LABEL_NAMES):
    print(f"{name:>12}", end="")
    for j in range(NUM_LABELS):
        print(f"{cm_after[i, j]:>10}", end="")
    print()
 
print(f"\nConfusion Matrix — 변화량 (양수=증가, 음수=감소):")
print(f"{'':>12}", end="")
for name in LABEL_NAMES:
    print(f"{name:>10}", end="")
print()
for i, name in enumerate(LABEL_NAMES):
    print(f"{name:>12}", end="")
    for j in range(NUM_LABELS):
        val = cm_diff[i, j]
        if val != 0:
            print(f"{val:>+10d}", end="")
        else:
            print(f"{'·':>10}", end="")
    print()
 
# --- 타겟 혼동축별 변화 ---
print("\n타겟 혼동축별 오분류 건수 변화:")
confusion_axes = [
    ("협박", "갈취"),
    ("갈취", "협박"),
    ("기타괴롭힘", "갈취"),
    ("갈취", "기타괴롭힘"),
    ("직장내괴롭힘", "기타괴롭힘"),
    ("기타괴롭힘", "직장내괴롭힘"),
    ("협박", "기타괴롭힘"),
    ("기타괴롭힘", "협박"),
]
print(f"{'실제→예측':>20} | {'전':>5} | {'후':>5} | {'변화':>5}")
print("-" * 45)
for true_name, pred_name in confusion_axes:
    ti = LABEL2ID[true_name]
    pi = LABEL2ID[pred_name]
    before_val = cm_before[ti, pi]
    after_val = cm_after[ti, pi]
    diff = after_val - before_val
    marker = "✓" if diff < 0 else ("✗" if diff > 0 else "=")
    print(f"{true_name:>8}→{pred_name:<10} | {before_val:>5} | {after_val:>5} | {diff:>+5d} {marker}")

In [ ]:
# ============================================================================
# 09. 하이퍼파라미터 민감도 분석 (Ablation)
# ============================================================================
# margin_threshold와 boost_per_hit의 조합별 성능 변화를 그리드로 확인
 
MARGIN_THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
BOOST_VALUES = [0.3, 0.5, 0.7, 1.0, 1.5]
 
ablation_results = []
 
for mt in MARGIN_THRESHOLDS:
    for bv in BOOST_VALUES:
        adj_preds, adj_log = apply_rule_postprocessing(
            logits=raw_logits,
            texts=val_texts,
            margin_threshold=mt,
            boost_per_hit=bv,
            max_boost=bv * 3,   # max_boost = 3 hits worth
        )
        f1m = f1_score(val_labels, adj_preds, average="macro")
        acc = np.mean(adj_preds == val_labels)
        n_changed = adj_log["n_changed"]
        ablation_results.append({
            "margin_threshold": mt,
            "boost_per_hit": bv,
            "macro_f1": round(f1m, 4),
            "accuracy": round(acc, 4),
            "delta_f1": round(f1m - baseline_f1_macro, 4),
            "n_changed": n_changed,
        })
 
df_ablation = pd.DataFrame(ablation_results)
 
# --- Pivot Table 출력 ---
pivot_f1 = df_ablation.pivot(
    index="margin_threshold",
    columns="boost_per_hit",
    values="delta_f1"
)
print("Ablation: Macro F1 변화량 (baseline 대비)")
print("행=margin_threshold, 열=boost_per_hit")
print("=" * 70)
print(pivot_f1.to_string(float_format="{:+.4f}".format))
 
# --- 최적 조합 ---
best_row = df_ablation.loc[df_ablation["macro_f1"].idxmax()]
print(f"\n최적 조합: margin={best_row['margin_threshold']}, "
      f"boost={best_row['boost_per_hit']}")
print(f"  Macro F1: {best_row['macro_f1']:.4f} (delta: {best_row['delta_f1']:+.4f})")
print(f"  변경 건수: {int(best_row['n_changed'])}건")
 
# --- 변경 건수 분포 ---
pivot_changes = df_ablation.pivot(
    index="margin_threshold",
    columns="boost_per_hit",
    values="n_changed"
)
print(f"\nAblation: 예측 변경 건수")
print(pivot_changes.to_string())

In [ ]:
# ============================================================================
# 10. 시각화
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
 
# --- 한글 폰트 ---
font_candidates = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
]
font_path = None
for fc in font_candidates:
    if os.path.exists(fc):
        font_path = fc
        break
if font_path:
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()
else:
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
 
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
 
# --- (a) Margin 분포 히스토그램 ---
ax = axes[0]
ax.hist(margins, bins=50, color="#4C78A8", edgecolor="white", alpha=0.8)
ax.axvline(x=0.15, color="#E45756", linestyle="--", linewidth=2, label="threshold=0.15")
ax.set_xlabel("top1-top2 margin")
ax.set_ylabel("샘플 수")
ax.set_title("(a) Softmax Margin 분포")
ax.legend()
 
# --- (b) Ablation Heatmap ---
ax = axes[1]
pivot_data = pivot_f1.values
im = ax.imshow(pivot_data, cmap="RdYlGn", aspect="auto",
               vmin=-0.01, vmax=0.01)
ax.set_xticks(range(len(BOOST_VALUES)))
ax.set_xticklabels([f"{v}" for v in BOOST_VALUES])
ax.set_yticks(range(len(MARGIN_THRESHOLDS)))
ax.set_yticklabels([f"{v}" for v in MARGIN_THRESHOLDS])
ax.set_xlabel("boost_per_hit")
ax.set_ylabel("margin_threshold")
ax.set_title("(b) Ablation: ΔMacro F1")
for i in range(len(MARGIN_THRESHOLDS)):
    for j in range(len(BOOST_VALUES)):
        ax.text(j, i, f"{pivot_data[i, j]:+.4f}",
                ha="center", va="center", fontsize=7,
                color="white" if abs(pivot_data[i, j]) > 0.005 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
 
# --- (c) 클래스별 F1 비교 (Before/After) ---
ax = axes[2]
from sklearn.metrics import f1_score as sk_f1
f1_before_list = []
f1_after_list = []
for i in range(NUM_LABELS):
    f1_before_list.append(sk_f1(val_labels == i, baseline_preds == i, zero_division=0))
    f1_after_list.append(sk_f1(val_labels == i, adjusted_preds == i, zero_division=0))
 
x = np.arange(NUM_LABELS)
width = 0.35
bars1 = ax.bar(x - width/2, f1_before_list, width, label="후처리 전", color="#4C78A8")
bars2 = ax.bar(x + width/2, f1_after_list, width, label="후처리 후", color="#E45756")
ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES, fontsize=9)
ax.set_ylabel("F1 Score")
ax.set_title("(c) 클래스별 F1 비교")
ax.legend()
ax.set_ylim(0.8, 1.0)
ax.grid(axis="y", alpha=0.3)
 
plt.tight_layout()
plt.savefig("dktc_rule_postprocessing_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("  → 저장: dktc_rule_postprocessing_analysis.png")

In [ ]:
# ============================================================================
# 11. 키워드 커버리지 분석 — 규칙이 실제 오분류를 얼마나 커버하는가
# ============================================================================
# 목적: 키워드 사전의 품질 평가
# 오분류 샘플에 대해 '정답 클래스의 키워드가 텍스트에 존재하는지' 확인
 
wrong_mask = baseline_preds != val_labels
wrong_indices = np.where(wrong_mask)[0]
 
print(f"전체 오분류 {len(wrong_indices)}건에 대한 키워드 커버리지 분석:\n")
 
coverage_stats = {"covered": 0, "not_covered": 0, "no_rule_for_class": 0}
per_class_coverage = {i: {"total": 0, "covered": 0} for i in range(NUM_LABELS)}
 
for idx in wrong_indices:
    true_label = val_labels[idx]
    text = val_texts[idx]
 
    per_class_coverage[true_label]["total"] += 1
 
    if true_label not in COMPILED_RULES:
        coverage_stats["no_rule_for_class"] += 1
        continue
 
    hits = count_keyword_hits(text, true_label)
    if hits > 0:
        coverage_stats["covered"] += 1
        per_class_coverage[true_label]["covered"] += 1
    else:
        coverage_stats["not_covered"] += 1
 
print(f"커버리지 요약:")
print(f"  키워드로 커버 가능  : {coverage_stats['covered']}건")
print(f"  키워드 미매칭      : {coverage_stats['not_covered']}건")
print(f"  규칙 미정의 클래스  : {coverage_stats['no_rule_for_class']}건 (일반대화)")
 
# 규칙이 정의된 클래스(0~3)에서의 커버리지율
rule_classes_total = coverage_stats['covered'] + coverage_stats['not_covered']
if rule_classes_total > 0:
    print(f"\n규칙 대상 클래스 오분류 커버리지: "
          f"{coverage_stats['covered']}/{rule_classes_total} "
          f"({coverage_stats['covered']/rule_classes_total*100:.1f}%)")
 
print(f"\n클래스별 오분류 커버리지:")
print(f"{'클래스':>10} | {'오분류':>6} | {'커버':>4} | {'커버율':>7}")
print("-" * 38)
for i, name in enumerate(LABEL_NAMES):
    total = per_class_coverage[i]["total"]
    covered = per_class_coverage[i]["covered"]
    rate = covered / total * 100 if total > 0 else 0
    print(f"{name:>10} | {total:>6} | {covered:>4} | {rate:>6.1f}%")
 
# --- 미커버 오분류 샘플 확인 (개선 방향 파악용) ---
print(f"\n미커버 오분류 샘플 (최대 10건 — 키워드 개선 참고용):")
n_shown = 0
for idx in wrong_indices:
    true_label = val_labels[idx]
    if true_label not in COMPILED_RULES:
        continue
    hits = count_keyword_hits(val_texts[idx], true_label)
    if hits == 0:
        pred_label = baseline_preds[idx]
        text_preview = val_texts[idx][:120].replace("\n", " ")
        print(f"  [{idx:4d}] 정답={LABEL_NAMES[true_label]:8s} 예측={LABEL_NAMES[pred_label]:8s} | {text_preview}...")
        n_shown += 1
        if n_shown >= 10:
            break

In [ ]:
# ============================================================================
# 12. 최적 설정으로 최종 후처리 적용 + Kaggle 제출용 함수
# ============================================================================
# ablation 결과에서 최적 파라미터를 선택한 뒤,
# test 데이터에도 동일하게 적용할 수 있는 함수를 정리한다.
 
# --- 최적 파라미터 (ablation 결과 기준) ---
# 주의: 아래 값은 ablation 결과를 보고 수동 설정할 것
BEST_MARGIN_THRESHOLD = best_row["margin_threshold"]
BEST_BOOST_PER_HIT = best_row["boost_per_hit"]
BEST_MAX_BOOST = BEST_BOOST_PER_HIT * 3
 
print(f"최종 후처리 파라미터:")
print(f"  margin_threshold : {BEST_MARGIN_THRESHOLD}")
print(f"  boost_per_hit    : {BEST_BOOST_PER_HIT}")
print(f"  max_boost        : {BEST_MAX_BOOST}")
 
 
def postprocess_for_submission(logits, texts):
    """
    Kaggle 제출용 후처리 함수.
    test 데이터의 raw logits와 원본 텍스트를 입력받아
    규칙 기반 보정된 예측을 반환.
 
    사용법:
        test_result = trainer.predict(test_dataset)
        test_logits = test_result.predictions
        test_preds = postprocess_for_submission(test_logits, test_texts)
    """
    adjusted_preds, _ = apply_rule_postprocessing(
        logits=logits,
        texts=texts,
        margin_threshold=BEST_MARGIN_THRESHOLD,
        boost_per_hit=BEST_BOOST_PER_HIT,
        max_boost=BEST_MAX_BOOST,
    )
    return adjusted_preds
 
 
# --- 최종 검증셋 성능 재확인 ---
final_preds, final_log = apply_rule_postprocessing(
    logits=raw_logits,
    texts=val_texts,
    margin_threshold=BEST_MARGIN_THRESHOLD,
    boost_per_hit=BEST_BOOST_PER_HIT,
    max_boost=BEST_MAX_BOOST,
)
 
final_f1 = f1_score(val_labels, final_preds, average="macro")
final_acc = np.mean(final_preds == val_labels)
 
print(f"\n최종 검증 성능:")
print(f"  Accuracy   : {final_acc:.4f}")
print(f"  Macro F1   : {final_f1:.4f}")
print(f"  변경 건수   : {final_log['n_changed']}")
print(f"  Delta F1   : {final_f1 - baseline_f1_macro:+.4f}")

In [ ]:
# ============================================================================
# 13. 결과 저장
# ============================================================================
 
# --- Ablation 결과 CSV ---
df_ablation.to_csv("dktc_rule_postprocessing_ablation.csv", index=False)
print("dktc_rule_postprocessing_ablation.csv 저장 완료")
 
# --- 키워드 규칙 JSON (재현성) ---
rules_export = {}
for class_name, patterns in KEYWORD_RULES.items():
    rules_export[class_name] = patterns
 
with open("dktc_keyword_rules_v1.json", "w", encoding="utf-8") as f:
    json.dump(rules_export, f, ensure_ascii=False, indent=2)
print("dktc_keyword_rules_v1.json 저장 완료")
 
# --- 후처리 설정 JSON ---
config_export = {
    "margin_threshold": float(BEST_MARGIN_THRESHOLD),
    "boost_per_hit": float(BEST_BOOST_PER_HIT),
    "max_boost": float(BEST_MAX_BOOST),
    "baseline_macro_f1": float(baseline_f1_macro),
    "postprocessed_macro_f1": float(final_f1),
    "delta_f1": float(final_f1 - baseline_f1_macro),
}
with open("dktc_postprocessing_config.json", "w") as f:
    json.dump(config_export, f, indent=2)
print("dktc_postprocessing_config.json 저장 완료")
 
print(f"\n=" * 70)
print(f"규칙 기반 후처리 분석 완료")
print(f"  Baseline Macro F1     : {baseline_f1_macro:.4f}")
print(f"  Postprocessed Macro F1: {final_f1:.4f}")
print(f"  Delta                 : {final_f1 - baseline_f1_macro:+.4f}")
print(f"=" * 70)

In [ ]:
# ============================================================================
# 14. 해석 및 다음 단계
# ============================================================================
# 이 셀은 코드가 아닌 분석 메모 용도
 
print("""
# ================================================================
# 해석 가이드
# ================================================================
#
# 1. 후처리 효과가 양수(+)인 경우:
#    - 키워드 규칙이 경계 샘플에서 올바른 방향으로 작동
#    - 해당 파라미터 조합을 Kaggle 제출에 사용 가능
#    - 단, 검증셋 과적합 가능성 있으므로 CV 결과와 교차 확인 필요
#
# 2. 후처리 효과가 0 또는 음수(-)인 경우:
#    - 키워드 사전이 불충분하거나 false positive가 많음
#    - 키워드 커버리지 분석(§11)에서 미커버 샘플 확인
#    - 키워드를 추가/수정한 뒤 재실행
#    - 효과가 계속 없으면 규칙 후처리보다 2단계 분류 우선 고려
#
# 3. 예측 변경 건수가 너무 많은 경우 (>30건):
#    - margin_threshold가 너무 크거나 boost가 너무 강함
#    - threshold를 줄이거나 boost를 낮출 것
#
# 4. 다음 단계 (런북 §8 기준):
#    - [ ] 최적 후처리 설정으로 Kaggle 제출 1회
#    - [ ] 후처리 없는 원본 모델도 동시 제출 (비교용)
#    - [ ] 미커버 오분류 패턴 기반으로 키워드 v2 작성
#    - [ ] 2단계 분류 프로토타입 시작
#    - [ ] 3-fold CV에서 후처리 효과 재검증
#
# ================================================================
""")